# Study Buddy

This notebook builds a simple **PDF question-answering app** using:

- **Gradio** for the interface
- **LangChain** for PDF loading and text splitting
- **FAISS** for vector search
- **Hugging Face models** for embeddings and answer generation

The workflow is:

1. Install the required libraries.
2. Load the models.
3. Upload a PDF.
4. Split the PDF into chunks.
5. Store the chunks in a FAISS vector database.
6. Ask questions and generate answers from the relevant chunks.


## 1. Install dependencies

Run this cell once in Colab or Jupyter if the packages are not already installed.

In [ ]:
!pip install -q gradio langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu transformers pypdf torch


## 2. Import libraries

These imports handle the UI, document loading, chunking, embeddings, vector search, and text generation.

In [ ]:
import gradio as gr
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


## 3. Load models and define global variables

This cell prepares:

- the computation device (`GPU` if available, otherwise `CPU`)
- the embedding model used to convert text chunks into vectors
- the text generation model used to answer questions
- a global variable that will hold the FAISS vector database after the PDF is processed


In [ ]:
print("Loading models...")

device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# This will store the indexed document after PDF processing.
vector_db = None

print(f"Using device: {device}")


## 4. Define the PDF processing function

This function:

1. accepts an uploaded PDF
2. loads the pages from the PDF
3. splits the text into smaller chunks
4. converts those chunks into embeddings
5. stores the result inside FAISS for retrieval


In [ ]:
def process_pdf(file):
    global vector_db

    if file is None:
        return "Error: Please upload a PDF first."

    try:
        loader = PyPDFLoader(file.name)
        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=200,
            chunk_overlap=50,
        )
        chunks = splitter.split_documents(documents)

        vector_db = FAISS.from_documents(chunks, embeddings)
        return f"Success: PDF split into {len(chunks)} chunks and indexed."
    except Exception as exc:
        return f"Error processing PDF: {exc}"


## 5. Define the response function

When the user asks a question, this function:

1. searches the FAISS index for the most relevant chunks
2. combines those chunks into a context block
3. builds a prompt using the context and the user question
4. sends the prompt to the FLAN-T5 model
5. returns the generated answer


In [ ]:
def respond(message, history):
    global vector_db

    if vector_db is None:
        return "Please upload and process a PDF in the left panel first."

    results = vector_db.similarity_search(message, k=3)
    context = "\n".join(result.page_content for result in results)

    prompt = (
        "Answer the question using the context below.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{message}\n\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_new_tokens=150)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


## 6. Build the Gradio interface

The interface has two parts:

- a left panel for PDF upload and processing
- a right panel for asking questions and viewing answers


In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Study Buddy")
    gr.Markdown("Upload a PDF, process it, and ask questions about its content.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Setup")
            file_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("Process Document", variant="primary")
            status = gr.Textbox(
                label="System Status",
                placeholder="Awaiting upload...",
                interactive=False,
            )

        with gr.Column(scale=2):
            gr.Markdown("### Chat")
            chat_ui = gr.ChatInterface(
                fn=respond,
                examples=["What is the main topic?", "Summarize the document"],
                cache_examples=False,
            )

    process_btn.click(fn=process_pdf, inputs=file_input, outputs=status)


## 7. Launch the app

Run this final cell to start the Gradio interface.

In [ ]:
demo.launch(debug=True)
